# Анализ морфологических биопсий

Данный блокнот анализирует данные морфологических биопсий пациентов, обследованных в разные временные точки с ординальными измерениями (шкалы 0-3 или 0-2).

## Дизайн исследования
- **Группы**: ОГ против КГ
- **Временные точки**: день операции, 6-7 сутки, 1-3 мес., 1 год
- **Локации**: полип, средняя нос.раковина
- **Поля**: 1-5 на биопсию
- **Параметры**: 9 морфологических характеристик (ординальные шкалы)

## Цели
1. Сравнить группы по фазам для локации "средняя нос.раковина"
2. Сравнить группы на этапе "день операции"
3. Определить предпочтительную группу лечения в целом и по фазам

## Статистический подход
- Непараметрические тесты для ординальных данных
- Расчет размера эффекта
- Коррекция множественных сравнений
- Визуализация различий между группами

## 1. Загрузка необходимых библиотек

In [ ]:
# Load required libraries
library(dplyr)        # Data manipulation
library(ggplot2)      # Data visualization
library(tidyr)        # Data reshaping
library(readxl)       # Reading Excel files
library(knitr)        # Table formatting
library(broom)        # Tidy statistical output
library(coin)         # Non-parametric tests
library(effsize)      # Effect size calculations
library(ordinal)      # Ordinal regression
library(MASS)         # Statistical functions
library(gridExtra)    # Arranging plots
library(RColorBrewer) # Color palettes
library(corrplot)     # Correlation plots
library(psych)        # Descriptive statistics

# Set options
options(scipen = 999)  # Prevent scientific notation
theme_set(theme_minimal())

print("Libraries loaded successfully!")

## 2. Загрузка и изучение данных

In [ ]:
# Load the biopsy data from specific Excel file and sheet
data_raw <- read_excel("C:\\Analysis\\OTOLARING\\Nidelko\\mucous20240825.xlsx", sheet = "данные")

print("Raw data loaded successfully!")
print(paste("Raw dimensions:", nrow(data_raw), "rows ×", ncol(data_raw), "columns"))
print("\nRaw column names:")
print(colnames(data_raw))

# Filter out unnecessary columns
data = data_raw %>% 
    dplyr::select("id", "группа", "локация", "этап", "поле", "круглоклеточная воспалительная инфильтрация", 
            "бокаловидные клетки", "отек", "реснички эпителия", "фиброз", "гиперплазия респираторного эпителия", 
            "плоскоклеточная метаплазия", "эозинофилы", "нейтрофилы"
            )

print("\nData structure:")
str(data)

# Define morphological parameters
morph_params <- c("круглоклеточная воспалительная инфильтрация", 
                  "бокаловидные клетки", 
                  "отек", 
                  "реснички эпителия", 
                  "фиброз", 
                  "гиперплазия респираторного эпителия", 
                  "плоскоклеточная метаплазия", 
                  "эозинофилы", 
                  "нейтрофилы")

print("\nMorphological parameters to analyze:")
print(morph_params)

## 3. Предобработка данных и агрегация

In [ ]:
# Data preprocessing and aggregation
# Aggregate data by median across fields (поле) for each patient, location, and phase
try({
    data_clean <- data %>%
        # Convert to proper factors
        mutate(
            группа = factor(группа, levels = c("КГ", "КГ1", "ОГ")),
            локация = factor(локация),
            этап = factor(этап, levels = c("день операции", "6-7 сутки", "1-3 мес.", "1 год")),
            id = factor(id)
        ) %>%
        # Group by patient, location, and phase to aggregate across fields
        group_by(id, группа, локация, этап) %>%
        summarise(
            across(all_of(morph_params), ~ median(.x, na.rm = TRUE)),
            .groups = "drop"
        )
    
    print("Data preprocessed and aggregated by median across fields")
    print(paste("Final dimensions:", nrow(data_clean), "rows ×", ncol(data_clean), "columns"))
})

# Check for missing values
try({
    missing_summary <- data_clean %>%
        summarise(across(everything(), ~ sum(is.na(.x))))
    print("\nMissing values summary:")
    print(missing_summary)
})

# Summary of cleaned data
print("\nSummary of key variables:")
try({
    print("Groups:")
    table(data_clean$группа)
    print("Locations:")
    table(data_clean$локация)
    print("Phases:")
    table(data_clean$этап)
    print("Patients per group:")
    data_clean %>% 
        distinct(id, группа) %>% 
        count(группа) %>%
        kable(col.names = c("Group", "N Patients"))
})

## 4. Описательная статистика

In [ ]:
# Descriptive statistics

# Function to calculate descriptive statistics for ordinal data
calc_ordinal_stats <- function(data, group_var, params) {
    tryCatch({
        data %>%
            group_by(!!sym(group_var)) %>%
            summarise(
                across(all_of(params), list(
                    median = ~ median(.x, na.rm = TRUE),
                    q25 = ~ quantile(.x, 0.25, na.rm = TRUE),
                    q75 = ~ quantile(.x, 0.75, na.rm = TRUE),
                    min = ~ min(.x, na.rm = TRUE),
                    max = ~ max(.x, na.rm = TRUE),
                    n = ~ sum(!is.na(.x))
                )),
                .groups = "drop"
            )
    }, error = function(e) {
        print(paste("Error in calc_ordinal_stats:", e$message))
        return(data.frame())
    })
}

# Overall descriptive statistics by group
print("=== OVERALL DESCRIPTIVE STATISTICS BY GROUP ===")
try({
    overall_stats <- calc_ordinal_stats(data_clean, "группа", morph_params)
    kable(overall_stats, digits = 2)
})

# Sample sizes by group, location, and phase
print("\n=== SAMPLE SIZES BY GROUP, LOCATION, AND PHASE ===")
try({
    sample_sizes <- data_clean %>%
        group_by(группа, локация, этап) %>%
        summarise(n = n(), .groups = "drop") %>%
        pivot_wider(names_from = группа, values_from = n, values_fill = 0)
    kable(sample_sizes)
})

# Descriptive statistics for средняя нос.раковина
print("\n=== DESCRIPTIVE STATISTICS FOR СРЕДНЯЯ НОС.РАКОВИНА ===")
try({
    nasal_data <- data_clean %>% filter(локация == "средняя нос.раковина")
    nasal_stats <- calc_ordinal_stats(nasal_data, "группа", morph_params)
    kable(nasal_stats, digits = 2)
})

# Descriptive statistics for день операции
print("\n=== DESCRIPTIVE STATISTICS FOR ДЕНЬ ОПЕРАЦИИ ===")
try({
    surgery_data <- data_clean %>% filter(этап == "день операции")
    surgery_stats <- calc_ordinal_stats(surgery_data, "группа", morph_params)
    kable(surgery_stats, digits = 2)
})

## 5. Анализ 1: Сравнение групп по фазам для средняя нос.раковина

In [ ]:
# Analysis 1: Group comparison by phase for средняя нос.раковина

# Function to perform Mann-Whitney U test with effect size
perform_mann_whitney <- function(group1_data, group2_data, param_name, comparison_name) {
    tryCatch({
        if (length(group1_data) < 3 || length(group2_data) < 3) {
            return(data.frame(
                parameter = param_name,
                comparison = comparison_name,
                n_kg = length(group1_data),
                n_og = length(group2_data),
                median_kg = median(group1_data, na.rm = TRUE),
                median_og = median(group2_data, na.rm = TRUE),
                p_value = NA,
                cliff_delta = NA,
                effect_size = "Insufficient data",
                stringsAsFactors = FALSE
            ))
        }
        
        # Check normality with Shapiro-Wilk test
        og_normal <- ifelse(length(group2_data) > 2, shapiro.test(group2_data)$p.value > 0.05, FALSE)
        kg_normal <- ifelse(length(group1_data) > 2, shapiro.test(group1_data)$p.value > 0.05, FALSE)
        
        # Use Mann-Whitney U test (more appropriate for ordinal data)
        test_result <- wilcox.test(group1_data, group2_data, exact = FALSE)
        
        # Cliff's delta effect size
        cliff_result <- cliff.delta(group1_data, group2_data)
        
        # Effect size interpretation
        abs_cliff <- abs(cliff_result$estimate)
        effect_interp <- case_when(
            abs_cliff < 0.147 ~ "Negligible",
            abs_cliff < 0.33 ~ "Small",
            abs_cliff < 0.474 ~ "Medium",
            TRUE ~ "Large"
        )
        
        return(data.frame(
            parameter = param_name,
            comparison = comparison_name,
            n_kg = length(group1_data),
            n_og = length(group2_data),
            median_kg = median(group1_data, na.rm = TRUE),
            median_og = median(group2_data, na.rm = TRUE),
            p_value = test_result$p.value,
            cliff_delta = cliff_result$estimate,
            effect_size = effect_interp,
            stringsAsFactors = FALSE
        ))
    }, error = function(e) {
        print(paste("Error in perform_mann_whitney for", param_name, ":", e$message))
        return(data.frame(
            parameter = param_name,
            comparison = comparison_name,
            n_kg = length(group1_data),
            n_og = length(group2_data),
            median_kg = median(group1_data, na.rm = TRUE),
            median_og = median(group2_data, na.rm = TRUE),
            p_value = NA,
            cliff_delta = NA,
            effect_size = "Error",
            stringsAsFactors = FALSE
        ))
    })
}

# Filter data for средняя нос.раковина
try({
    nasal_data <- data_clean %>% filter(локация == "средняя нос.раковина")
    print("=== ANALYSIS 1: GROUP COMPARISON BY PHASE FOR СРЕДНЯЯ НОС.РАКОВИНА ===")
})

# Results storage
nasal_results <- data.frame()

# Analysis for each phase
try({
    for (phase in levels(nasal_data$этап)) {
        phase_data <- nasal_data %>% filter(этап == phase)
        
        if (nrow(phase_data) > 0) {
            print(paste("\n--- Phase:", phase, "---"))
            
            for (param in morph_params) {
                kg_data <- phase_data %>% filter(группа == "КГ") %>% pull(!!sym(param))
                og_data <- phase_data %>% filter(группа == "ОГ") %>% pull(!!sym(param))
                
                result <- perform_mann_whitney(kg_data, og_data, param, phase)
                nasal_results <- rbind(nasal_results, result)
            }
        }
    }
})

# Apply multiple comparison correction (FDR)
try({
    nasal_results$p_adjusted <- p.adjust(nasal_results$p_value, method = "fdr")
    
    # Add significance indicators
    nasal_results <- nasal_results %>%
        mutate(
            significance = case_when(
                is.na(p_adjusted) ~ "",
                p_adjusted < 0.001 ~ "***",
                p_adjusted < 0.01 ~ "**",
                p_adjusted < 0.05 ~ "*",
                p_adjusted < 0.1 ~ ".",
                TRUE ~ ""
            ),
            better_group = case_when(
                is.na(cliff_delta) ~ "N/A",
                cliff_delta > 0 ~ "КГ better",
                cliff_delta < 0 ~ "ОГ better",
                TRUE ~ "No difference"
            )
        )
})

print("\nResults for средняя нос.раковина by phase:")
try({
    kable(nasal_results %>% 
          select(comparison, parameter, n_kg, n_og, median_kg, median_og, 
                 p_value, p_adjusted, significance, cliff_delta, effect_size, better_group),
          digits = 4)
})

## 6. Анализ 2: Сравнение групп для день операции

In [ ]:
# Analysis 2: Group comparison for день операции

# Filter data for день операции
try({
    surgery_data <- data_clean %>% filter(этап == "день операции")
    print("=== ANALYSIS 2: GROUP COMPARISON FOR ДЕНЬ ОПЕРАЦИИ ===")
})

# Results storage
surgery_results <- data.frame()

# Analysis for each location on surgery day
try({
    for (location in levels(surgery_data$локация)) {
        location_data <- surgery_data %>% filter(локация == location)
        
        if (nrow(location_data) > 0) {
            print(paste("\n--- Location:", location, "---"))
            
            for (param in morph_params) {
                kg_data <- location_data %>% filter(группа == "КГ") %>% pull(!!sym(param))
                og_data <- location_data %>% filter(группа == "ОГ") %>% pull(!!sym(param))
                
                result <- perform_mann_whitney(kg_data, og_data, param, location)
                surgery_results <- rbind(surgery_results, result)
            }
        }
    }
})

# Apply multiple comparison correction (FDR)
try({
    surgery_results$p_adjusted <- p.adjust(surgery_results$p_value, method = "fdr")
    
    # Add significance indicators
    surgery_results <- surgery_results %>%
        mutate(
            significance = case_when(
                is.na(p_adjusted) ~ "",
                p_adjusted < 0.001 ~ "***",
                p_adjusted < 0.01 ~ "**",
                p_adjusted < 0.05 ~ "*",
                p_adjusted < 0.1 ~ ".",
                TRUE ~ ""
            ),
            better_group = case_when(
                is.na(cliff_delta) ~ "N/A",
                cliff_delta > 0 ~ "КГ better",
                cliff_delta < 0 ~ "ОГ better",
                TRUE ~ "No difference"
            )
        )
})

print("\nResults for день операции by location:")
try({
    kable(surgery_results %>% 
          select(comparison, parameter, n_kg, n_og, median_kg, median_og, 
                 p_value, p_adjusted, significance, cliff_delta, effect_size, better_group),
          digits = 4)
})

## 7. Комплексное резюме результатов

In [ ]:
# Comprehensive Results Summary

# Combine all results
try({
    all_results <- bind_rows(
        nasal_results %>% mutate(analysis = "Phase comparison (средняя нос.раковина)"),
        surgery_results %>% mutate(analysis = "Location comparison (день операции)")
    )
    
    print("=== COMPREHENSIVE RESULTS SUMMARY ===")
    print("Significance codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1")
    print("Effect size (Cliff's delta): <0.147 negligible, <0.33 small, <0.474 medium, ≥0.474 large")
})

# Summary of significant results
try({
    significant_results <- all_results %>%
        filter(!is.na(p_adjusted) & p_adjusted < 0.05) %>%
        arrange(p_adjusted)
    
    print(paste("\n=== SIGNIFICANT RESULTS (FDR-corrected p < 0.05) ==="))
    print(paste("Number of significant comparisons:", nrow(significant_results), "out of", nrow(all_results %>% filter(!is.na(p_adjusted)))))
    
    if (nrow(significant_results) > 0) {
        kable(significant_results %>% 
              select(analysis, comparison, parameter, better_group, 
                     median_kg, median_og, p_adjusted, cliff_delta, effect_size),
              digits = 4,
              col.names = c("Analysis", "Comparison", "Parameter", "Better Group", 
                           "Median КГ", "Median ОГ", "Adj. P-value", "Cliff's δ", "Effect Size"))
    } else {
        print("No statistically significant differences found after multiple comparison correction.")
    }
})

# Summary by analysis type
print("\n=== SUMMARY BY ANALYSIS TYPE ===")
try({
    summary_by_analysis <- all_results %>%
        filter(!is.na(p_adjusted)) %>%
        group_by(analysis) %>%
        summarise(
            total_comparisons = n(),
            significant_p05 = sum(p_adjusted < 0.05),
            significant_p01 = sum(p_adjusted < 0.01),
            large_effects = sum(effect_size == "Large", na.rm = TRUE),
            medium_effects = sum(effect_size == "Medium", na.rm = TRUE),
            og_better = sum(better_group == "ОГ better", na.rm = TRUE),
            kg_better = sum(better_group == "КГ better", na.rm = TRUE),
            .groups = "drop"
        )
    
    kable(summary_by_analysis)
})

## 8. Визуализация данных

In [ ]:
# Data Visualization

# Colors for groups
group_colors <- c("КГ" = "#E69F00", "ОГ" = "#56B4E9")

# Plot 1: Boxplots for средняя нос.раковина by phase
create_boxplot_grid <- function(data, title_prefix) {
    tryCatch({
        plots <- list()
        
        for (i in 1:length(morph_params)) {
            param <- morph_params[i]
            
            p <- data %>%
                ggplot(aes_string(x = "этап", y = paste0("`", param, "`"), fill = "группа")) +
                geom_boxplot(alpha = 0.7, outlier.alpha = 0.6) +
                scale_fill_manual(values = group_colors) +
                labs(title = paste(param), y = "Score", x = "Phase", fill = "Group") +
                theme_minimal() +
                theme(
                    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
                    plot.title = element_text(size = 10),
                    legend.position = "none"
                )
            
            plots[[i]] <- p
        }
        
        # Add legend to the last plot
        plots[[length(plots)]] <- plots[[length(plots)]] + theme(legend.position = "bottom")
        
        return(plots)
    }, error = function(e) {
        print(paste("Error creating boxplot grid:", e$message))
        return(list())
    })
}

# Boxplots for средняя нос.раковина
print("Creating boxplots for средняя нос.раковина by phase...")
try({
    nasal_plots <- create_boxplot_grid(nasal_data, "средняя нос.раковина")
    if (length(nasal_plots) > 0) {
        grid.arrange(grobs = nasal_plots, ncol = 3, 
                     top = "Group Comparison by Phase (средняя нос.раковина)")
    }
})

# Boxplots for день операции
print("Creating boxplots for день операции by location...")
try({
    surgery_plots <- list()

    for (i in 1:length(morph_params)) {
        param <- morph_params[i]
        
        p <- surgery_data %>%
            ggplot(aes_string(x = "локация", y = paste0("`", param, "`"), fill = "группа")) +
            geom_boxplot(alpha = 0.7, outlier.alpha = 0.6) +
            scale_fill_manual(values = group_colors) +
            labs(title = paste(param), y = "Score", x = "Location", fill = "Group") +
            theme_minimal() +
            theme(
                axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
                plot.title = element_text(size = 10),
                legend.position = "none"
            )
        
        surgery_plots[[i]] <- p
    }

    # Add legend to the last plot
    surgery_plots[[length(surgery_plots)]] <- surgery_plots[[length(surgery_plots)]] + 
        theme(legend.position = "bottom")

    grid.arrange(grobs = surgery_plots, ncol = 3,
                 top = "Group Comparison by Location (день операции)")
})

In [ ]:
# Heatmap of effect sizes
print("Creating effect size heatmap...")

# Prepare data for heatmap
try(heatmap_data <- all_results %>%
    filter(!is.na(cliff_delta)) %>%
    select(analysis, comparison, parameter, cliff_delta, p_adjusted) %>%
    mutate(
        significant = ifelse(p_adjusted < 0.05, "*", ""),
        comparison_param = paste(comparison, parameter, sep = " - ")
    )) 


if (try(nrow(heatmap_data)) > 0) {
    try(p_heatmap <- heatmap_data %>%
        ggplot(aes(x = parameter, y = comparison, fill = cliff_delta)) +
        geom_tile(color = "white", size = 0.5) +
        geom_text(aes(label = significant), color = "white", size = 6, fontface = "bold") +
        scale_fill_gradient2(
            low = "blue", mid = "white", high = "red",
            midpoint = 0, name = "Cliff's δ",
            limits = c(-1, 1)
        ) +
        facet_wrap(~ analysis, scales = "free_y", ncol = 1) +
        labs(
            title = "Effect Sizes (Cliff's Delta) Heatmap",
            subtitle = "* indicates FDR-corrected p < 0.05",
            x = "Morphological Parameter",
            y = "Comparison"
        ) +
        theme_minimal() +
        theme(
            axis.text.x = element_text(angle = 45, hjust = 1),
            strip.text = element_text(face = "bold")
        ))

    try(print(p_heatmap))
} else {
    print("No data available for heatmap")
}

# Median values comparison plot
print("Creating median values comparison plot...")

try(median_comparison <- all_results %>%
    filter(!is.na(median_kg) & !is.na(median_og)) %>%
    select(analysis, comparison, parameter, median_kg, median_og, p_adjusted) %>%
    pivot_longer(cols = c(median_kg, median_og), names_to = "group", values_to = "median") %>%
    mutate(
        group = recode(group, "median_kg" = "КГ", "median_og" = "ОГ"),
        significant = ifelse(p_adjusted < 0.05, "Significant", "Not significant")
    ))

if (try(nrow(median_comparison)) > 0) {
    try(p_medians <- median_comparison %>%
        ggplot(aes(x = parameter, y = median, fill = group)) +
        geom_col(position = "dodge", alpha = 0.8) +
        scale_fill_manual(values = group_colors) +
        facet_grid(analysis ~ comparison, scales = "free") +
        labs(
            title = "Median Values Comparison",
            x = "Morphological Parameter",
            y = "Median Score",
            fill = "Group"
        ) +
        theme_minimal() +
        theme(
            axis.text.x = element_text(angle = 45, hjust = 1),
            strip.text = element_text(size = 8)
        )
    )

    try(print(p_medians))
}

## 9. Расширенный анализ: Ординальная логистическая регрессия

In [ ]:
# Advanced Analysis: Ordinal Logistic Regression
# This provides odds ratios and confidence intervals

print("=== ORDINAL LOGISTIC REGRESSION ANALYSIS ===")

# Function to perform ordinal logistic regression
perform_ordinal_regression <- function(data, param, comparison_name) {
    tryCatch({
        # Prepare data
        model_data <- data %>%
            select(группа, !!sym(param)) %>%
            filter(!is.na(!!sym(param))) %>%
            mutate(
                outcome = factor(!!sym(param), ordered = TRUE),
                группа = relevel(factor(группа), ref = "КГ")  # КГ as reference
            )
        
        if (nrow(model_data) < 10 || length(unique(model_data$outcome)) < 2) {
            return(data.frame(
                parameter = param,
                comparison = comparison_name,
                n = nrow(model_data),
                or = NA,
                or_lower = NA,
                or_upper = NA,
                p_value = NA,
                note = "Insufficient data",
                stringsAsFactors = FALSE
            ))
        }
        
        # Fit ordinal logistic regression
        model <- polr(outcome ~ группа, data = model_data, Hess = TRUE)
        
        # Extract results
        summary_table <- summary(model)
        or <- exp(coef(model))
        ci <- exp(confint(model))
        p_value <- summary_table$coefficients[1, 4]
        
        return(data.frame(
            parameter = param,
            comparison = comparison_name,
            n = nrow(model_data),
            or = or,
            or_lower = ci[1],
            or_upper = ci[2],
            p_value = p_value,
            note = "Success",
            stringsAsFactors = FALSE
        ))
        
    }, error = function(e) {
        print(paste("Error in ordinal regression for", param, "in", comparison_name, ":", e$message))
        return(data.frame(
            parameter = param,
            comparison = comparison_name,
            n = ifelse(exists("model_data"), nrow(model_data), 0),
            or = NA,
            or_lower = NA,
            or_upper = NA,
            p_value = NA,
            note = paste("Error:", e$message),
            stringsAsFactors = FALSE
        ))
    })
}

# Ordinal regression for средняя нос.раковина by phase
ordinal_results <- data.frame()

print("Performing ordinal logistic regression for средняя нос.раковина by phase...")
try({
    for (phase in levels(nasal_data$этап)) {
        phase_data <- nasal_data %>% filter(этап == phase)
        
        if (nrow(phase_data) > 0) {
            print(paste("Processing phase:", phase))
            
            for (param in morph_params) {
                result <- perform_ordinal_regression(phase_data, param, paste("Phase:", phase))
                ordinal_results <- rbind(ordinal_results, result)
            }
        }
    }
})

# Ordinal regression for день операции by location
print("Performing ordinal logistic regression for день операции by location...")
try({
    for (location in levels(surgery_data$локация)) {
        location_data <- surgery_data %>% filter(локация == location)
        
        if (nrow(location_data) > 0) {
            print(paste("Processing location:", location))
            
            for (param in morph_params) {
                result <- perform_ordinal_regression(location_data, param, paste("Location:", location))
                ordinal_results <- rbind(ordinal_results, result)
            }
        }
    }
})

# Apply multiple comparison correction
try({
    ordinal_results$p_adjusted <- p.adjust(ordinal_results$p_value, method = "fdr")
    print("Multiple comparison correction applied successfully")
})

# Add significance indicators
try({
    ordinal_results <- ordinal_results %>%
        mutate(
            significance = case_when(
                is.na(p_adjusted) ~ "",
                p_adjusted < 0.001 ~ "***",
                p_adjusted < 0.01 ~ "**",
                p_adjusted < 0.05 ~ "*",
                p_adjusted < 0.1 ~ ".",
                TRUE ~ ""
            ),
            or_interpretation = case_when(
                is.na(or) ~ "N/A",
                or > 1 ~ "ОГ higher odds",
                or < 1 ~ "ОГ lower odds",
                TRUE ~ "No difference"
            )
        )
    print("Significance indicators added successfully")
})

print("\nOrdinal Logistic Regression Results:")
print("Odds Ratios (OR) > 1: ОГ group has higher odds of higher scores")
print("Odds Ratios (OR) < 1: ОГ group has lower odds of higher scores")

try({
    successful_results <- ordinal_results %>% filter(note == "Success")
    
    if (nrow(successful_results) > 0) {
        kable(successful_results %>%
              select(comparison, parameter, n, or, or_lower, or_upper, 
                     p_value, p_adjusted, significance, or_interpretation),
              digits = 3,
              col.names = c("Comparison", "Parameter", "N", "OR", "OR Lower", "OR Upper",
                           "P-value", "Adj. P-value", "Sig.", "Interpretation"))
    } else {
        print("No successful ordinal regression models were fitted.")
        print("This may be due to insufficient data or issues with data structure.")
    }
    
    # Summary of model fitting success
    success_summary <- ordinal_results %>%
        count(note) %>%
        mutate(percentage = round(100 * n / sum(n), 1))
    
    print("\nModel fitting summary:")
    kable(success_summary, col.names = c("Result", "Count", "Percentage (%)"))
})

## 10. Клиническое резюме и рекомендации

In [ ]:
# Clinical Summary and Recommendations

print("=== CLINICAL SUMMARY AND RECOMMENDATIONS ===")

# Generate summary statistics
total_patients <- length(unique(data_clean$id))
total_comparisons_made <- nrow(all_results %>% filter(!is.na(p_adjusted)))
significant_findings <- nrow(all_results %>% filter(!is.na(p_adjusted) & p_adjusted < 0.05))

cat("STUDY OVERVIEW:\n")
cat("- Total patients analyzed:", total_patients, "\n")
cat("- Total statistical comparisons:", total_comparisons_made, "\n")
cat("- Significant findings (FDR-corrected p < 0.05):", significant_findings, "\n")
cat("- Percentage of significant findings:", round(100 * significant_findings / total_comparisons_made, 1), "%\n\n")

# Summary by group preference
og_better_count <- sum(all_results$better_group == "ОГ better" & 
                       all_results$p_adjusted < 0.05, na.rm = TRUE)
kg_better_count <- sum(all_results$better_group == "КГ better" & 
                       all_results$p_adjusted < 0.05, na.rm = TRUE)

cat("GROUP PERFORMANCE SUMMARY (significant findings only):\n")
cat("- ОГ group performed better:", og_better_count, "times\n")
cat("- КГ group performed better:", kg_better_count, "times\n\n")

# Identify most affected parameters
if (nrow(significant_results) > 0) {
    param_summary <- significant_results %>%
        count(parameter, sort = TRUE) %>%
        head(5)
    
    cat("MOST FREQUENTLY DIFFERENT PARAMETERS:\n")
    for (i in 1:nrow(param_summary)) {
        cat(paste0(i, ". ", param_summary$parameter[i], " (", param_summary$n[i], " significant comparisons)\n"))
    }
    cat("\n")
}

# Effect size summary
large_effects <- all_results %>%
    filter(effect_size == "Large" & !is.na(p_adjusted)) %>%
    nrow()

medium_effects <- all_results %>%
    filter(effect_size == "Medium" & !is.na(p_adjusted)) %>%
    nrow()

cat("EFFECT SIZE SUMMARY:\n")
cat("- Large effect sizes:", large_effects, "\n")
cat("- Medium effect sizes:", medium_effects, "\n\n")

cat("STATISTICAL METHODS USED:\n")
cat("1. Mann-Whitney U tests for group comparisons (appropriate for ordinal data)\n")
cat("2. Cliff's delta for effect size estimation\n")
cat("3. FDR correction for multiple comparisons\n")
cat("4. Ordinal logistic regression for odds ratios\n")
cat("5. Median aggregation across fields for each patient\n\n")

cat("INTERPRETATION GUIDELINES:\n")
cat("- P-values: FDR-corrected to control false discovery rate\n")
cat("- Effect sizes (Cliff's δ): <0.147 negligible, 0.147-0.33 small, 0.33-0.474 medium, ≥0.474 large\n")
cat("- Odds ratios: >1 means ОГ group has higher odds of higher scores\n")
cat("- Clinical significance should be considered alongside statistical significance\n\n")

cat("RECOMMENDATIONS FOR FURTHER ANALYSIS:\n")
cat("1. Consider longitudinal analysis if multiple time points per patient are available\n")
cat("2. Investigate interactions between time and treatment group\n")
cat("3. Adjust for baseline characteristics if available\n")
cat("4. Consider minimal clinically important differences for each parameter\n")
cat("5. Validate findings in independent cohorts if possible\n")